# Generació de frases

Produeix `lab/outputs/es/frases/dataset_entitats_whisper.jsonl`: per cada entitat
seleccionada, frases en diversos estils periodístics amb dues versions.

| camp | què és |
|---|---|
| `raw_text` | ortografia real — **ground truth** per entrenar Whisper |
| `tts_text` | la mateixa frase amb la fonètica del diccionari aplicada — el que llegeix OmniVoice |

**Fase 1** dedueix un context per entitat (sembrat del que ja va retornar la validació de
grafia). **Fase 2** genera les frases: el model escriu **només `raw_text`**. **Fase 3**
construeix `tts_text` aplicant el diccionari final, sense cap crida a cap model.

Crides, lots i prompts: `src/llm.py`. Diccionari i aplicació: `src/phonetics.py`.

In [ ]:
import concurrent.futures
import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/llm.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import llm
import phonetics

IDIOMA_TTS = "Castellano"
MODEL = "gpt-4.1-mini"  # el mateix que la resta del pipeline
client, proveidor = llm.client_per_model(MODEL)

# --- Entrades ---
ENT = ROOT / "lab/entitats/es"
RUTA_ENTITATS = ENT / "entidades_candidatas.json"
RUTA_DICCIONARI = phonetics.ruta_diccionari(IDIOMA_TTS, "final")

# --- Sortides (totes sota lab/outputs/es/frases) ---
DIR_FRASES = ROOT / "lab/outputs/es/frases"
DIR_FRASES.mkdir(parents=True, exist_ok=True)
ARCHIVO_AUDITORIA = DIR_FRASES / "contextos_auditables.json"
OUTPUT_FILE = DIR_FRASES / "dataset_entitats_whisper.jsonl"
ARCHIVO_DESCARTES = DIR_FRASES / "frases_descartadas.jsonl"

entitats_seleccionades = list(json.loads(RUTA_ENTITATS.read_text("utf-8")))
if not entitats_seleccionades:
    raise RuntimeError(f"No hi ha entitats seleccionades a {RUTA_ENTITATS}")

if not RUTA_DICCIONARI.exists():
    raise RuntimeError(f"No existeix el diccionari final: {RUTA_DICCIONARI}")
diccionari = phonetics.carregar(RUTA_DICCIONARI)

# El diccionari final nomes ha de contenir overrides d'entitats seleccionades.
claus_seleccionades = {phonetics.clau(e) for e in entitats_seleccionades}
if fora := [g for g in diccionari.entrades if phonetics.clau(g) not in claus_seleccionades]:
    raise RuntimeError(f"El diccionari final conte overrides no seleccionats: {fora[:10]}")

if sense_revisar := [g for g, e in diccionari.entrades.items() if not e.get("revisat")]:
    print(f"AVIS: {len(sense_revisar)}/{len(diccionari)} overrides sense revisio manual. "
          f"Revisa'ls a dictionary.ipynb abans de generar el dataset definitiu.")
    print(f"  {', '.join(sense_revisar[:8])}{'...' if len(sense_revisar) > 8 else ''}")

print(f"Entitats: {len(entitats_seleccionades)} | diccionari final: {len(diccionari)} "
      f"overrides ({RUTA_DICCIONARI.name}) | model {MODEL} ({proveidor})")

## Fase 1: context de cada entitat

Es sembra amb el tipo i el motivo que la validació de grafia ja va pagar
(entidades_fuente_{a,b}_validadas.json); només es demana al model el que hi falta.

Sortida: contextos_auditables.json -> revisable a mà abans de la Fase 2.

In [9]:
# Llavor: el `contexto` que la validacio de grafia ja va retornar per entitat. Es
# reaprofita en comptes de tornar-ho a comprar.
#
# Abans aqui es feia servir `motivo`, que es la justificacio de la decisio d'ORTOGRAFIA
# ("...grafía correcta según contexto deportivo"): el 45% mencionava tildes o grafies i
# un 14% no deia res mes ("Japón: País, nombre propio con tilde"), soroll inutil o
# confusionari per a qui ha d'escriure la frase. Ara el prompt de validacio torna els
# dos camps per separat i aqui nomes es llegeix el que toca.
vives = set(entitats_seleccionades)
contextos = {}
for nom in ("entidades_fuente_a_validadas.json", "entidades_fuente_b_validadas.json"):
    ruta = ENT / nom
    if not ruta.exists():
        continue
    dades = json.loads(ruta.read_text("utf-8"))
    for v in dades.get("validadas", []) + dades.get("rechazadas", []):
        grafia = v.get("grafia_correcta")
        context = (v.get("contexto") or "").strip()
        if grafia in vives and context and v.get("tipo") != "NO_ENTIDAD":
            contextos.setdefault(grafia, context)

if ARCHIVO_AUDITORIA.exists():   # revisions manuals previes manen
    contextos.update(json.loads(ARCHIVO_AUDITORIA.read_text("utf-8")))

pendents = [e for e in entitats_seleccionades if e not in contextos]
print(f"Fase 1: {len(contextos)} contextos sembrats de la validacio | {len(pendents)} a demanar")

if pendents:
    nous, meta = llm.processar_per_lots(
        pendents, 10, client, MODEL, llm.SYSTEM_CONTEXT, llm.SCHEMA_CONTEXT,
        "context_entitats", "entidades", "contexto", etiqueta="Context")
    contextos.update(nous)
    if meta["cost_usd"]:
        print(f"  cost: ${meta['cost_usd']:.4f}")

ARCHIVO_AUDITORIA.write_text(json.dumps(contextos, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n{len(contextos)} contextos -> {ARCHIVO_AUDITORIA}")
print("REVISIO MANUAL: corregeix els contextos equivocats abans de la Fase 2.")

Fase 1: 154 contextos sembrats de la validacio | 7 a demanar
Context 1/1 (7 entitats, gpt-4.1-mini)...
  cost: $0.0004

161 contextos -> /media/dd3/sintetic-dataset/lab/outputs/frases/contextos_auditables.json
REVISIO MANUAL: corregeix els contextos equivocats abans de la Fase 2.


## Fase 2 — generació de frases

Una crida per (entitat, estil). texto_tts ja porta totes les entitats reescrites.

In [ ]:
ESTILS = [
    "Titular de última hora (frases cortas y directas)",
    "Crónica detallada del corresponsal (frases más largas y descriptivas)",
    #"Entradilla del presentador en plató (tono formal e introductorio)",
    "Declaraciones en una rueda de prensa o debate (estilo más hablado)",
    #"Noticia breve de sección de impacto (tono urgente)",
]
FRASES_PER_BATCH = 5 #10
MAX_WORKERS = 4

# El model nomes escriu `texto`: la fonetica l'aplica la Fase 3 amb el diccionari.
SYSTEM_FRASES = llm.system_frases(IDIOMA_TTS)


def normalitzar(text):
    return re.sub(r"\s+", " ", text).strip().lower()


def generar_lot(entitat, context, estil):
    """Una crida: `FRASES_PER_BATCH` frases d'una entitat en un estil."""
    user = (f"Entidad objetivo: '{entitat}'\n"
            f"Contexto: {context}\n"
            f"Estilo periodístico de este lote: {estil}\n"
            f"Genera {FRASES_PER_BATCH} frases.")
    obj, _meta = llm.crida_amb_reintents(client, MODEL, SYSTEM_FRASES, user,
                                         llm.SCHEMA_FRASES, "frases", max_tokens=4096)
    return obj["frases"]


# Estat previ: dedup de frases i recompte per (entitat, estil) per poder rellancar.
frases_vistes, per_combo = set(), Counter()
if OUTPUT_FILE.exists():
    for linia in OUTPUT_FILE.read_text("utf-8").splitlines():
        if linia.strip():
            r = json.loads(linia)
            frases_vistes.add(normalitzar(r["raw_text"]))
            per_combo[(r["entity"], r["style"])] += 1

tasques = [(e, contextos[e], s) for e in entitats_seleccionades if e in contextos
           for s in ESTILS if per_combo[(e, s)] < FRASES_PER_BATCH]
print(tasques[:10])
print(f"Fase 2: {len(tasques)} crides pendents "
      f"({len(entitats_seleccionades) * len(ESTILS) - len(tasques)} combinacions ja fetes)")

In [ ]:

stats = Counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futurs = {executor.submit(generar_lot, e, c, s): (e, s) for e, c, s in tasques}
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out, \
         open(ARCHIVO_DESCARTES, "a", encoding="utf-8") as f_desc:
        for n, futur in enumerate(concurrent.futures.as_completed(futurs), start=1):
            entitat, estil = futurs[futur]
            try:
                frases = futur.result()
            except Exception as e:
                print(f"  ERROR [{entitat} / {estil[:20]}]: {e}")
                stats["error_lot"] += 1
                continue

            for frase in frases:
                text = frase.get("texto", "")
                # Prova que la frase parla de l'entitat demanada: la primera paraula
                # de l'entitat hi ha de ser (el model la pot flexionar, no substituir).
                arrel = (phonetics.clau(entitat).split() or [""])[0]
                if arrel and arrel not in phonetics.clau(text):
                    motiu = "no_conte_entitat"
                elif normalitzar(text) in frases_vistes:
                    motiu = "duplicada"
                else:
                    frases_vistes.add(normalitzar(text))
                    # `tts_text` es queda igual que `raw_text` fins que la Fase 3 hi
                    # apliqui el diccionari. Es desa igualment perque el fitxer sigui
                    # valid encara que la generacio s'interrompi.
                    f_out.write(json.dumps({"entity": entitat, "raw_text": text,
                                            "tts_text": text, "style": estil},
                                           ensure_ascii=False) + "\n")
                    stats["ok"] += 1
                    continue
                stats[motiu] += 1
                f_desc.write(json.dumps({"entity": entitat, "style": estil,
                                         "texto": text, "motivo": motiu},
                                        ensure_ascii=False) + "\n")
            if n % 25 == 0 or n == len(futurs):
                print(f"  {n}/{len(futurs)} lots | {stats['ok']} frases desades")

print(f"\nFase 2 acabada: {dict(stats)}")
print(f"Dataset -> {OUTPUT_FILE}")
print(f"Descartades (auditoria) -> {ARCHIVO_DESCARTES}")
print("\nSeguent: Fase 3 -- sense ella, `tts_text` encara es igual que `raw_text`.")

## Fase 3 — fonètica

Aplica el diccionari final sobre `raw_text` i expandeix xifres i símbols. Sense LLM.

Una entitat que no és al diccionari es queda tal com s'escriu: és el comportament
correcte, perquè `dictionary.ipynb` només hi posa overrides quan la grafia crua es
llegiria malament. Si al dataset hi apareixen entitats que sí que necessitarien override
i no en tenen, la manera d'arreglar-ho és afegir-les a la llista d'entitats i tornar a
passar `dictionary.ipynb` — no tocar-ho aquí.

In [ ]:
# Fase 3 — `tts_text` = diccionari final aplicat sobre `raw_text` + xifres expandides.
# Determinista i idempotent: es pot rellançar sobre el mateix fitxer les vegades que calgui.
GLOSSARI = diccionari.pla()
regex_glossari = phonetics.compilar_regex(GLOSSARI)

registres = [json.loads(l) for l in OUTPUT_FILE.read_text("utf-8").splitlines() if l.strip()]
comptador = Counter()
for r in registres:
    text, aplicades = phonetics.aplicar_diccionari(r["raw_text"], GLOSSARI, regex_glossari)
    r["tts_text"] = phonetics.expandir_xifres(text)
    comptador.update(phonetics.clau(a) for a in aplicades)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for r in registres:
        f.write(json.dumps({"entity": r["entity"], "raw_text": r["raw_text"],
                            "tts_text": r["tts_text"], "style": r["style"]},
                           ensure_ascii=False) + "\n")

n_diff = sum(1 for r in registres if r["tts_text"] != r["raw_text"])
print(f"{len(registres)} frases | {n_diff} amb fonètica aplicada | {len(GLOSSARI)} overrides al diccionari")
print("\nOverrides més aplicats:")
for ent, n in comptador.most_common(15):
    print(f"  {n:5d}x  {ent}")
print("\n--- MOSTRA ---")
for r in [x for x in registres if x["tts_text"] != x["raw_text"]][:4]:
    print(f"\n[{r['entity']}]")
    print(f"  raw: {r['raw_text']}")
    print(f"  tts: {r['tts_text']}")

## Verificació

Tres comprovacions que han de donar zero. La segona és la que garanteix que no s'ha
colat res del model dins de `tts_text`.

In [ ]:
# Verificació final.
a_vigilar = {g: f for g, f in GLOSSARI.items() if not phonetics.es_identitat(g, f)}
registres = [json.loads(l) for l in OUTPUT_FILE.read_text("utf-8").splitlines() if l.strip()]

# 1) Entitats del glossari que segueixen en cru a `tts_text`. Es compara amb una regex
#    sensible a majúscules: amb `re.IGNORECASE`, 'Aemet' casava amb el patró d'`AEMET` i
#    la comprovació antiga donava per fallats justament els overrides de caixa.
crues = Counter()
exemples = {}
for r in registres:
    for g, f in a_vigilar.items():
        rx = re.compile(phonetics.patro_entitat(g))
        if rx.search(r["raw_text"]) and rx.search(r["tts_text"]) and f not in r["tts_text"]:
            crues[g] += 1
            exemples.setdefault(g, r["tts_text"])

# 2) `tts_text` ha de ser exactament el que dona aplicar el diccionari sobre `raw_text`.
no_deterministes = [r for r in registres
                    if r["tts_text"] != phonetics.expandir_xifres(
                        phonetics.aplicar_diccionari(r["raw_text"], GLOSSARI, regex_glossari)[0])]

# 3) Frases d'entitats que ja no són a `entidades_candidatas.json`. Passa quan el
#    round-trip es torna a executar i mou el llindar: la Fase 2 és reprenible i afegeix
#    les que falten, però no treu les que sobren.
orfes = Counter(r["entity"] for r in registres if r["entity"] not in set(entitats_seleccionades))

print(f"Frases: {len(registres)} | overrides vigilats: {len(a_vigilar)}")
print(f"1. entitats del glossari en cru           : {len(crues)}")
for g, n in crues.most_common():
    print(f"     {n:5d}x {g!r} (hauria de ser {a_vigilar[g]!r})\n           {exemples[g][:100]}")
print(f"2. frases amb `tts_text` no determinista  : {len(no_deterministes)}")
for r in no_deterministes[:5]:
    print(f"     {r['raw_text'][:80]}")
print(f"3. frases d'entitats ja no seleccionades  : {sum(orfes.values())} ({len(orfes)} entitats)")
for ent, n in orfes.most_common(10):
    print(f"     {n:5d}x {ent!r}")
if orfes:
    print("     -> esborra'n les línies del .jsonl o torna a afegir l'entitat a la llista.")

if not (crues or no_deterministes):
    print("\nOK: `tts_text` és una funció pura de `raw_text` i del diccionari.")
print(f"\nSegüent: generate_voices_environments.ipynb (llegeix {OUTPUT_FILE.name})")